# RVC v2 Inference Server
**Thesis:** Integrating Speech-to-Text and RVC-Based Voice Conversion into an AI Assistant  
**Author:** Nguyen Hoang Ngoc Bao — 24MSE23204  

Maps to **Section 7 deliverable 3**:  
> *A Colab notebook that runs the RVC inference server and exposes it as a public HTTPS endpoint via cloudflared.*

Also maps to **Technique T2** (Section 6.3):  
> *RVC inference is moved off the local Python 3.12 host onto a Colab T4 GPU exposed via a cloudflared tunnel.*

**Before running:**
1. Complete `train_rvc.ipynb` first — it exports `<speaker>.pth` + `<speaker>.index` to Drive
2. Set `Runtime > Change runtime type > GPU (T4)`
3. Edit the `CONFIGURATION` cell (at minimum confirm `SPEAKER_NAME`)
4. Run all cells, then copy the printed `RVC_ENDPOINT` URL into your Flask `.env`

**API contract** (matches `voice/rvc.py` in the Flask app):
- `POST /convert`  — multipart: `audio` (MP3/WAV file), `pitch` (int), `index_rate` (float) → returns WAV bytes
- `GET  /health`   — returns `{"status": "ok"}` with HTTP 200


In [ ]:
# ── A. CONFIGURATION ─────────────────────────────────────────────────────────
# Must match the SPEAKER_NAME used in train_rvc.ipynb.

SPEAKER_NAME = "speaker_vi_female"   # same as train_rvc.ipynb
SERVER_PORT  = 7860
PITCH        = 0      # semitone shift (0 = no pitch change)
INDEX_RATE   = 0.75   # FAISS blend (0.0 = ignore index, 1.0 = full retrieval)
F0_METHOD    = "rmvpe"               # must match training
PROTECT      = 0.33   # protect voiceless consonants from over-conversion

DRIVE_ROOT   = "/content/drive/MyDrive/rvc_training"
MODEL_PTH    = f"{DRIVE_ROOT}/models/{SPEAKER_NAME}.pth"
MODEL_INDEX  = f"{DRIVE_ROOT}/models/{SPEAKER_NAME}.index"

print("Configuration:")
print(f"  speaker    : {SPEAKER_NAME}")
print(f"  model      : {MODEL_PTH}")
print(f"  index      : {MODEL_INDEX}")
print(f"  port       : {SERVER_PORT}")
print(f"  pitch      : {PITCH} semitones")
print(f"  index_rate : {INDEX_RATE}")
print(f"  f0_method  : {F0_METHOD}")


In [ ]:
# ── B. MOUNT DRIVE + INSTALL DEPENDENCIES ────────────────────────────────────
from google.colab import drive
import os

drive.mount("/content/drive")

assert os.path.exists(MODEL_PTH), (
    f"Model not found: {MODEL_PTH}\n"
    "Run train_rvc.ipynb first to train and export the model."
)
assert os.path.exists(MODEL_INDEX), (
    f"Index not found: {MODEL_INDEX}\n"
    "Run train_rvc.ipynb first to build and export the FAISS index."
)
print(f"✓  Model  : {os.path.getsize(MODEL_PTH)/1e6:.1f} MB")
print(f"✓  Index  : {os.path.getsize(MODEL_INDEX)/1e6:.1f} MB")

!pip install -q rvc-python flask pydub soundfile
print("\n✓  Dependencies installed.")


In [ ]:
# ── C. GPU CHECK ─────────────────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU (T4) and reconnect."
    )

print(f"✓  GPU  : {torch.cuda.get_device_name(0)}")
print(f"   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ── D. LOAD RVC MODEL ─────────────────────────────────────────────────────────
from rvc_python.infer import RVCInference

print(f"Loading RVC model: {SPEAKER_NAME} …")
rvc = RVCInference(device="cuda:0")
rvc.load_model(MODEL_PTH, MODEL_INDEX)
print(f"✓  Model loaded: {SPEAKER_NAME}")


In [ ]:
# ── E. FLASK INFERENCE SERVER ─────────────────────────────────────────────────
# Implements the API contract expected by voice/rvc.py in the Flask app:
#   POST /convert  (multipart: audio file + pitch + index_rate) → WAV bytes
#   GET  /health   → {"status": "ok"}
import io, os, tempfile, threading, time
from flask import Flask, request, jsonify, send_file

server = Flask(__name__)


@server.get("/health")
def health():
    return jsonify({"status": "ok", "model": SPEAKER_NAME})


@server.post("/convert")
def convert():
    if "audio" not in request.files:
        return jsonify({"error": "no audio file"}), 400

    audio_bytes = request.files["audio"].read()
    pitch       = int(request.form.get("pitch",      PITCH))
    index_rate  = float(request.form.get("index_rate", INDEX_RATE))

    # Write input to temp file
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as fin:
        fin.write(audio_bytes)
        in_path = fin.name

    out_path = in_path.replace(".mp3", "_rvc.wav")

    try:
        t0 = time.perf_counter()
        rvc.infer_file(
            input_path=in_path,
            output_path=out_path,
            f0_up_key=pitch,
            f0_method=F0_METHOD,
            index_rate=index_rate,
            protect=PROTECT,
        )
        latency_ms = round((time.perf_counter() - t0) * 1000)
        print(f"[RVC] converted in {latency_ms} ms  "
              f"pitch={pitch}  index_rate={index_rate}")

        return send_file(
            out_path,
            mimetype="audio/wav",
            as_attachment=False,
        )
    except Exception as exc:
        print(f"[RVC] Error: {exc}")
        return jsonify({"error": str(exc)}), 500
    finally:
        try: os.unlink(in_path)
        except: pass
        # out_path cleaned up after send_file completes (Flask streams it)


def _run_server():
    server.run(host="0.0.0.0", port=SERVER_PORT, debug=False)


t = threading.Thread(target=_run_server, daemon=True)
t.start()
time.sleep(1.5)
print(f"✓  RVC server listening on port {SERVER_PORT}")


In [ ]:
# ── F. CLOUDFLARED TUNNEL ─────────────────────────────────────────────────────
# Thesis Section 7 deliverable 3: "exposes it as a public HTTPS endpoint via cloudflared"
# The printed URL becomes RVC_ENDPOINT in the Flask app's environment.
import os, re, subprocess, threading, time

# Download cloudflared binary
if not os.path.exists("/content/cloudflared"):
    !wget -q \
        https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
        -O /content/cloudflared
    !chmod +x /content/cloudflared
    print("✓  cloudflared downloaded.")
else:
    print("✓  cloudflared already present.")

# Start tunnel and capture the public URL
_tunnel_url = []


def _run_tunnel():
    proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel",
         "--url", f"http://localhost:{SERVER_PORT}",
         "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        print(line, end="")
        m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
        if m and not _tunnel_url:
            _tunnel_url.append(m.group(0))


threading.Thread(target=_run_tunnel, daemon=True).start()

print("Starting cloudflared tunnel … (waiting up to 30 s)")
for _ in range(30):
    if _tunnel_url:
        break
    time.sleep(1)

if not _tunnel_url:
    raise RuntimeError("Tunnel URL not detected. Check cloudflared output above.")

RVC_ENDPOINT = _tunnel_url[0]
print(f"\n✓  Tunnel active: {RVC_ENDPOINT}")


In [ ]:
# ── G. PRINT RVC_ENDPOINT (copy this into your Flask app) ────────────────────
# The endpoint must remain reachable as long as this notebook is running.
# DO NOT stop or restart the runtime — cloudflared will assign a new URL.

print("=" * 62)
print("  RVC_ENDPOINT (set this in your Flask app environment)")
print("=" * 62)
print()
print(f"  {RVC_ENDPOINT}")
print()
print("PowerShell (Windows — local Flask dev):")
print(f"  $env:RVC_ENDPOINT = \"{RVC_ENDPOINT}\"")
print()
print("Bash / Linux:")
print(f"  export RVC_ENDPOINT=\"{RVC_ENDPOINT}\"")
print()
print(".env file:")
print(f"  RVC_ENDPOINT={RVC_ENDPOINT}")
print()
print("=" * 62)
print()

# Quick self-test: hit /health via the tunnel
import urllib.request, json, time
time.sleep(2)   # give tunnel a moment to stabilise
try:
    with urllib.request.urlopen(f"{RVC_ENDPOINT}/health", timeout=10) as r:
        body = json.loads(r.read())
    print(f"✓  /health check passed: {body}")
except Exception as exc:
    print(f"⚠️  /health check failed: {exc}")
    print("   The server may still be starting — retry in a few seconds.")
print()
print("Keep this notebook running — the endpoint goes offline when stopped.")
